In [27]:
''' getting important libraries which is used during our Model building
    from xgboost  we imported XGBClassifier,
    and then we evaluted our model using various metrics from sklearn.metrics like
    accuracy_score,classification_report'''
import networkx as nx
from node2vec import Node2Vec
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,classification_report

In [16]:
'''making a DF pipeline to our cleaned data set
   known as 'delhivery_cleaned_for_XGB.csv'  please not
   that it is created below 
   after label encoding of ftl to 1 and carting to 0
'''
df=pd.read_csv('delhivery_cleaned_for_XGB.csv')
df.columns

Index(['Unnamed: 0', 'data', 'trip_creation_time', 'route_schedule_uuid',
       'route_type', 'trip_uuid', 'source_center', 'source_name',
       'destination_center', 'destination_name', 'od_start_time',
       'od_end_time', 'start_scan_to_end_scan', 'is_cutoff', 'cutoff_factor',
       'cutoff_timestamp', 'actual_distance_to_destination', 'actual_time',
       'osrm_time', 'osrm_distance', 'factor', 'segment_actual_time',
       'segment_osrm_time', 'segment_osrm_distance', 'segment_factor',
       'log_segment_factor', 'log_segment_osrm_distance',
       'log_segment_osrm_time', 'log_segment_actual_time', 'log_factor',
       'chokepoint_flag', 'segment_name', 'log_actual_time',
       'log_start_scan_to_end_scan', 'log_cutoff_factor',
       'log_actual_distance_to_destination', 'route_type_encoded'],
      dtype='object')

In [17]:
df.head()

,Unnamed: 0,data,trip_creation_time,route_schedule_uuid,route_type,trip_uuid,source_center,source_name,destination_center,destination_name,...,log_segment_osrm_time,log_segment_actual_time,log_factor,chokepoint_flag,segment_name,log_actual_time,log_start_scan_to_end_scan,log_cutoff_factor,log_actual_distance_to_destination,route_type_encoded
0,0,training,20-09-2018 02:35,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Khambhat_MotvdDPP_D (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),...,2.484907,2.708050,0.820981,1,Khambhat_MotvdDPP_D (Gujarat) -> Khambhat_Motv...,2.708050,4.465908,2.302585,2.436737,0
1,1,training,20-09-2018 02:35,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Khambhat_MotvdDPP_D (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),...,2.302585,2.397895,0.788457,0,Khambhat_MotvdDPP_D (Gujarat) -> Khambhat_Motv...,3.218876,4.465908,2.944439,2.992569,0
2,2,training,20-09-2018 02:35,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Khambhat_MotvdDPP_D (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),...,2.079442,2.833213,0.887303,1,Khambhat_MotvdDPP_D (Gujarat) -> Khambhat_Motv...,3.713572,4.465908,3.332205,3.354709,0
3,3,training,20-09-2018 02:35,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Khambhat_MotvdDPP_D (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),...,2.564949,3.091042,0.936093,1,Khambhat_MotvdDPP_D (Gujarat) -> Khambhat_Motv...,4.143135,4.465908,3.610918,3.614103,0
4,4,training,20-09-2018 02:35,thanos::sroute:eb7bfc78-b351-4c0e-a951-fa3d5c3...,Carting,trip-153741093647649320,IND388121AAA,Khambhat_MotvdDPP_D (Gujarat),IND388620AAB,Khambhat_MotvdDPP_D (Gujarat),...,1.791759,1.945910,0.934309,0,Khambhat_MotvdDPP_D (Gujarat) -> Khambhat_Motv...,4.234107,4.465908,3.688879,3.698484,0


In [18]:
'''getting a set of  unique destination center
   getting a set of  unique source center 
   we first applied unique() to get unique list of dst_center then applied set()
   to get a set of destination_hubs,source_hubs
   then since source_hubs,destination_hubs are set 
   we get the union of both of them
   to get a set of unique nodes
'''
destination_center = set(df['destination_center'].unique())
source_center = set(df['source_center'].unique())
nodes = source_center.union(destination_center)

In [19]:
 '''
    with our target columns 'log_segment_actual_time' 
    since our data is already divided into 'training' and 'test'  we creating
    train_df and test_df base on the logic that column df['data'] == 'training'
    data frame column == training for train_df and test_df=='test
'''
test_df = df[df['data'] =='test'].copy()
train_df = df[df['data']=='training'].copy()

In [20]:
''' since we have to capture weights of the 
    edges using the median actual-vs-OSRM we
    first groupby our data using source_name,destination name 
    then we applied aggregation function of median to both
    ratio of segment_factor and median of base_distance
    we reset index to 0,1,2,3,4.... for our data in row wise
'''
network_edge = df.groupby(['source_name','destination_name']).agg(
    median_ratio = ('segment_factor','median'),
    base_distance = ('segment_osrm_distance','median')
).reset_index()

In [21]:
'''we then created a directed graph 
   and then we use iteration to fill our Graph edges 
   with the weight using network_edge data frame and G.add_edge()
   which we have created earlier we gave it information 
   source_name and destination_name 
   then we printed total number of edges and number of nodes
'''
G = nx.DiGraph()
for index,row in network_edge.iterrows():
    G.add_edge(
        row['source_name'],
        row['destination_name'],
        weight = row['median_ratio'],
        distance = row['base_distance']
    )
print('Graph Created Successfuly')
print(f'Number of nodes {G.number_of_nodes()}')
print(f'Number of edges {G.number_of_edges()}')

Graph Created Successfuly
Number of nodes 1475
Number of edges 1491


In [22]:
''' 
    in node2vec we then creating our randoms walks of dimension 32 and each walk length is 15
    we have taken dimension=32 because it lies in goldie-lock zone since if we try to do 64 it makes 
    computation expensively and no significant changes in the Model Performance and we have taken 
    walk_length=15 so that each walker can go upto 15 nodes and by
    num_walks=4 we mean to say each node is performing 4 walks. and p=1.0 is for DFS and q=0.5 BFS
'''
node2vec = Node2Vec(G,dimensions=32,walk_length=15,num_walks=4,p=1.0,q=0.5)
'''
this is where our node2vec random walk is converted in to embeddings with window =10 ,min
window=10: This sets the context window, meaning the model looks at up to 10 nodes ahead and 10 nodes behind the 
target node in a random walk sequence to learn their structural relationships and network neighborhoods.

min_count=1: This ensures that every single node is included in the embedding space, 
preventing the model from discarding rare or low-degree nodes that only appear once in the random walks.

batch_words=4: This acts as the batch size for the underlying training loop, pushing the 
text/walk data to the operating system's threads 4 words (nodes) at a time to manage memory 
cache efficiency during gradient updates . and finally we get them saved in dictionary hub_embeddings.
'''
print('Generating Structural graph embeddings for hubs')
graph_model = node2vec.fit(window=10,min_count=1,batch_words=4)
hub_embeddings = {node:graph_model.wv[node] for node in G.nodes()}

Computing transition probabilities:   0%|          | 0/1475 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 4/4 [00:00<00:00, 15.05it/s]


Generating Structural graph embeddings for hubs


In [23]:
embedding_dim = 32
fallback_vector = np.zeros(embedding_dim)
def get_hub_features(df,embedding_dict):
     ''' 
    this function is basically to add the hub embeddings for 
    dst_hub embeddings src_hub_embeddings to our 
    orignaldataframe.
    we do the iteration in our rows of dataframe then
    in the src = we get the src_name
    in the dst = we get the dst_name
    in the src_vec we get the value of embeddings similarly for dest_vec 
    and then we finally add then store src_df,dst_df and then finally 
    we return the concatenetted dataframe which is horizontaaly saved.
    '''
    source_vector = []
    dest_vector = []
    for _,row in df.iterrows():
        src = row['source_name']
        dst = row['destination_name']
        src_vec = embedding_dict.get(src)
        dst_vec = embedding_dict.get(dst)
        source_vector.append(src_vec)
        dest_vector.append(dst_vec)
    src_df = pd.DataFrame(source_vector,columns=[f'src_emb_{i}' for i in range(embedding_dim)],index=df.index)
    dst_df = pd.DataFrame(dest_vector,columns=[f'dst_emb_{i}' for i in range(embedding_dim)],index=df.index)
    print(src_df)
    return pd.concat([df,src_df,dst_df],axis=1)
print('Mapping graph features back to tabular data splits')
''' we then get a new datframe which has 
    train_df columns and its hub embedding 
    similarly for test_enhance ''' 
train_enhanced = get_hub_features(train_df,hub_embeddings)
test_enhanced = get_hub_features(test_df,hub_embeddings)

Mapping graph features back to tabular data splits
        src_emb_0  src_emb_1  src_emb_2  src_emb_3  src_emb_4  src_emb_5  \
0        0.380237  -0.478285   0.458658  -0.883503   0.973496  -1.134388   
1        0.380237  -0.478285   0.458658  -0.883503   0.973496  -1.134388   
2        0.380237  -0.478285   0.458658  -0.883503   0.973496  -1.134388   
3        0.380237  -0.478285   0.458658  -0.883503   0.973496  -1.134388   
4        0.380237  -0.478285   0.458658  -0.883503   0.973496  -1.134388   
...           ...        ...        ...        ...        ...        ...   
142497  -0.100157  -0.031894   0.142192  -0.010877   0.087737  -0.051898   
142498  -0.100157  -0.031894   0.142192  -0.010877   0.087737  -0.051898   
142499  -0.100157  -0.031894   0.142192  -0.010877   0.087737  -0.051898   
142500  -0.100157  -0.031894   0.142192  -0.010877   0.087737  -0.051898   
142501  -0.100157  -0.031894   0.142192  -0.010877   0.087737  -0.051898   

        src_emb_6  src_emb_7  src_em

In [15]:
#encoding

In [24]:
''' we imported XGBRegressor 
    the selected the base features which has
    significant correlation with our target column 'segment_actual_time'
'''
base_features = [
    'cutoff_factor',
    'actual_distance_to_destination',
    'osrm_time',
    'osrm_distance',
    'segment_osrm_distance',
    'segment_osrm_time',
    'route_type_encoded'
]
''' 
    then we are getting the column which are for embeddings using if in their column 'src_emb_','dst_emb_'
    then we saved the ultimate feature which contain base_features + embedding_feature
    then we splitted them on the basis of train_enhanced and test_enhanced.
'''
target_col = 'route_type_encoded'
embedding_feature = [col for col in train_enhanced.columns if 'src_emb_' in col or 'dst_emb_' in col]
ultimate_features = base_features + embedding_feature
X_train = train_enhanced[ultimate_features]
y_train = train_enhanced[target_col]
X_test = test_enhanced[ultimate_features]
y_test = test_enhanced[target_col]

In [28]:

'''
    now we are building our XGBClassifier
'''
model = XGBClassifier(random_state=42)
#then we fitted our X_train and y_train
model.fit(X_train, y_train)


# 3. Make predictions on the test data
y_pred = model.predict(X_test)

# 4. Evaluate the model
''' we then check our goodness of classification using various metrices like
    precison,recall,f1_score,
    accuracy.
'''
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
#printing our classification report
print(classification_report(y_test, y_pred))

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13014
           1       1.00      1.00      1.00     26362

    accuracy                           1.00     39376
   macro avg       1.00      1.00      1.00     39376
weighted avg       1.00      1.00      1.00     39376



In [30]:
'''
    similarly doing it with LogisticRegression
    we fitted it

'''
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train, y_train)
print("Logistic Regression Accuracy:", accuracy_score(y_test, baseline.predict(X_test)))

Logistic Regression Accuracy: 0.9999746038195856


In [ ]:
''' thus since we used embeddings thats  why we are getting such high accuaracy'''